# 01 韓國輸電網路布局（OSM 資料）

改寫自上游 `notebooks/validation/validation_nigeria.ipynb`（1.1–1.2 節）與
`notebooks/validation/validation_namibia.ipynb`（1.1–1.2 節），合併成一本韓國版。

內容：
1. **1.1** OSM clean 資料的變電所與線路上色地圖
2. **1.2** 線路長度按電壓等級統計（OSM raw 對 OSM clean 自比）

**與原版的主要差異**

| 項目 | 非洲原版 | 韓國版 |
|---|---|---|
| 電壓分級 | 66 / 132 / 220 / 330 / 350 / 400 kV | **154 / 345 / 765 kV**，其餘歸「其他」 |
| 外部對照來源 | World Bank 非洲電網 geojson | **已刪除**（韓國不適用），改為 OSM raw 對 clean 自比 |
| 長度投影 | EPSG:3857 | **EPSG:5179**（Korea 2000）；3857 在北緯 37° 會高估約 24% |
| 國家過濾 | `country == "NA"` / `"NG"` 字串比對 | 全網（資料本身只有 KR） |

## 0. 環境設定

In [ ]:
import sys
import warnings
from pathlib import Path

_here = Path.cwd()
for cand in (_here, _here / "notebooks_kr", _here.parent):
    if (cand / "_kr_common.py").exists():
        sys.path.insert(0, str(cand))
        break

import _kr_common as K

K.setup_matplotlib()

import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.validation import make_valid

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("專案根目錄：", K.ROOT)
print("資源目錄  ：", K.RES)
print("輸出目錄  ：", K.FIG_DIR)

## 1.1 OSM clean 資料的網路布局

### 1.1.1 載入 clean OSM 資料

In [ ]:
df_sub = gpd.read_file(K.OSM_CLEAN_SUBSTATIONS)
df_sub["geometry"] = df_sub["geometry"].apply(make_valid)

df_lines = gpd.read_file(K.OSM_CLEAN_LINES)
df_lines["geometry"] = df_lines["geometry"].apply(make_valid)

print(f"變電所：{len(df_sub):,} 筆")
print(f"線路　：{len(df_lines):,} 筆")
print("國家欄位：", df_sub.country.unique(), df_lines.country.unique())

### 1.1.2 電壓等級分級

韓國輸電系統的電壓等級是 **154 / 345 / 765 kV** 三級。OSM 資料中還混有
145 / 180 / 250 / 354 / 380 kV 等非標準值（多半是標註誤差或跨國慣例殘留），
一律歸入「其他」，不硬套非洲版的 66/132/220/330/350/400 kV 級距。

In [ ]:
print("線路電壓原始值 [V]　：", sorted(df_lines.voltage.unique()))
print("變電所電壓原始值 [V]：", sorted(df_sub.voltage.unique()))

df_lines["voltage_bin"] = df_lines.voltage.apply(K.voltage_bin)
df_sub["voltage_bin"] = df_sub.voltage.apply(K.voltage_bin)

bin_order = ["154 kV", "345 kV", "765 kV", "其他"]

summary = pd.DataFrame({
    "線路數": df_lines.voltage_bin.value_counts().reindex(bin_order, fill_value=0),
    "變電所數": df_sub.voltage_bin.value_counts().reindex(bin_order, fill_value=0),
})
summary

### 1.1.3 網路布局地圖

投影固定用 `ccrs.PlateCarree()`，範圍固定在 `KR_EXTENT`（含濟州）。
底圖用 `country_shapes.geojson`，不依賴外部圖磚服務，確保離線也能重跑。

In [ ]:
# cimgt.OSM() 圖磚需連外。改成 True 可換成 OSM 底圖；預設 False 以保證離線可重現。
USE_OSM_TILES = False

country = gpd.read_file(K.COUNTRY_SHAPES)

fig, ax = plt.subplots(
    figsize=(9, 10), subplot_kw={"projection": ccrs.PlateCarree()}
)
ax.set_extent(K.KR_EXTENT, crs=ccrs.PlateCarree())

if USE_OSM_TILES:
    from cartopy.io.img_tiles import OSM
    ax.add_image(OSM(), 7)
else:
    country.plot(
        ax=ax, facecolor="whitesmoke", edgecolor="#999999",
        linewidth=0.6, transform=ccrs.PlateCarree(), zorder=0,
    )

for b in bin_order:
    sel_l = df_lines[df_lines.voltage_bin == b]
    if len(sel_l):
        sel_l.plot(
            ax=ax, color=K.KR_VOLTAGE_COLORS[b], linewidth=1.5 if b != "其他" else 0.6,
            alpha=0.85, transform=ccrs.PlateCarree(), zorder=2,
            label=f"{b} 線路 ({len(sel_l):,})",
        )

for b in bin_order:
    sel_s = df_sub[df_sub.voltage_bin == b]
    if len(sel_s):
        ax.scatter(
            sel_s.geometry.x, sel_s.geometry.y, s=6,
            color=K.KR_VOLTAGE_COLORS[b], edgecolor="none", alpha=0.55,
            transform=ccrs.PlateCarree(), zorder=3,
            label=f"{b} 變電所 ({len(sel_s):,})",
        )

ax.set_title("韓國輸電網路布局（OSM clean 資料，依電壓等級上色）", fontsize=14, pad=12)
ax.legend(loc="upper left", fontsize=9, framealpha=0.9, markerscale=2)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4)
gl.top_labels = False
gl.right_labels = False

K.savefig("01_fig1_osm_clean_network_KR")
plt.show()

### 1.1.4 布局說明

上圖可以看出 OSM clean 資料重現了韓國輸電網的主要結構：沿海岸與首都圈密集的 154 kV
配電骨幹、連接發電基地到負載中心的 345 kV 幹線，以及少數 765 kV 超高壓幹線
（新安城—新加平—新太白一帶）。濟州島因為靠 HVDC 與本土相連，在 OSM 交流線路
資料中呈現為獨立的一小片網路。

**沒有官方對照圖**：非洲版在這裡放的是 TCN / NamPower 年報的官方電網圖做並排比對。
本 repo 內沒有 KEPCO 官方電網圖的授權檔案，因此不放並排圖，也不自行描述官方圖的內容。

## 1.2 線路長度按電壓等級統計

### 1.2.1 投影選擇：為什麼不用 EPSG:3857

上游 notebook 用 `EPSG:3857`（Web Mercator）算長度。Web Mercator 的尺度失真是
`1/cos(緯度)`，在韓國（北緯 33–38.6°）約高估 **20–28%**。非洲案例靠近赤道時誤差還小，
韓國則不可忽略，因此改用 **EPSG:5179（Korea 2000 / Unified CS）**，這是韓國國家標準的
橫麥卡托投影。下面直接把兩者算出來比較，把差異講清楚。

In [ ]:
len_3857 = df_lines.to_crs(epsg=3857).geometry.length.sum() / 1000
len_5179 = df_lines.to_crs(K.KR_EQUAL_AREA_CRS).geometry.length.sum() / 1000

print(f"EPSG:3857（Web Mercator）幾何總長：{len_3857:>10,.0f} km")
print(f"EPSG:5179（Korea 2000）幾何總長　：{len_5179:>10,.0f} km")
print(f"高估倍率：{len_3857 / len_5179:.3f}　（理論值 1/cos(37.5°) = {1 / np.cos(np.radians(37.5)):.3f}）")

### 1.2.2 OSM clean 資料的線路長度

clean 資料每條線有明確的 `circuits`（迴路數）欄位。迴路長度 = 幾何長度 × 迴路數。

註：clean 資料自帶的 `length` 欄位在本次產出中數值全為 0，因此一律由幾何重算，
不使用該欄位。

In [ ]:
df_lines["geom_km"] = df_lines.to_crs(K.KR_EQUAL_AREA_CRS).geometry.length / 1000
df_lines["circuit_km"] = df_lines["geom_km"] * df_lines["circuits"]

clean_by_v = df_lines.groupby("voltage_bin")[["geom_km", "circuit_km"]].sum()
clean_by_v = clean_by_v.reindex(bin_order, fill_value=0.0)
clean_by_v.columns = ["幾何長度_km", "迴路長度_km"]
clean_by_v.loc["合計"] = clean_by_v.sum()
clean_by_v.round(0)

### 1.2.3 OSM raw 資料的線路長度

raw 資料的 `tags.voltage` 是字串，且同一條幾何線可能標多個電壓
（例如 `'345000;154000;154000'` 表示同一走廊掛了不同電壓的迴路）。
**取其中最高電壓做為該條線的代表電壓**，並在下面標明這是簡化慣例。

`tags.cables` 缺值時依上游慣例補 3（三相單迴路），迴路數 = cables / 3。

In [ ]:
df_raw = gpd.read_file(K.OSM_RAW_LINES)


def max_voltage(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    parts = [p.strip() for p in str(val).split(";") if p.strip()]
    nums = []
    for p in parts:
        try:
            nums.append(float(p))
        except ValueError:
            continue
    return max(nums) if nums else np.nan


df_raw["voltage_max"] = df_raw["tags.voltage"].apply(max_voltage)
df_raw["voltage_bin"] = df_raw["voltage_max"].apply(K.voltage_bin)

cables = pd.to_numeric(df_raw["tags.cables"], errors="coerce").fillna(3.0)
df_raw["geom_km"] = df_raw.to_crs(K.KR_EQUAL_AREA_CRS).geometry.length / 1000
df_raw["circuit_km"] = df_raw["geom_km"] * cables / 3.0

raw_by_v = df_raw.groupby("voltage_bin")[["geom_km", "circuit_km"]].sum()
raw_by_v = raw_by_v.reindex(bin_order, fill_value=0.0)
raw_by_v.columns = ["幾何長度_km", "迴路長度_km"]
raw_by_v.loc["合計"] = raw_by_v.sum()

n_missing_v = int(df_raw["voltage_max"].isna().sum())
n_missing_c = int(pd.to_numeric(df_raw["tags.cables"], errors="coerce").isna().sum())
print(f"raw 線路共 {len(df_raw):,} 筆，其中電壓缺標 {n_missing_v:,} 筆、cables 缺標 {n_missing_c:,} 筆（cables 缺值補 3）")
raw_by_v.round(0)

### 1.2.4 raw 對 clean 比較

取代原版的 World Bank 非洲電網對照（韓國不適用）。

這張表要讀成「清理前後的一致性檢查」，不是「資料損失率」——因為結果是
**clean 略高於 raw**（迴路長度 30,568 vs 28,806 km，+6%），而不是一般預期的清理後變少。
拆開看原因：

- **迴路數幾乎一致**：clean 的 `circuits` 合計 5,819，raw 的 `cables/3` 合計 5,797，
  差不到 0.4%。所以差異不是來自迴路數被重新認定。
- **差異來自幾何長度**：clean 14,996 km 對 raw 14,281 km（+715 km）。可直接歸因的部分是
  多電壓走廊被拆成數條 clean 線（raw 有 41 筆 `tags.voltage` 含分號、合計 180 km；
  clean 側有 164 筆共用同一個 raw `line_id` 前綴）；其餘來自清理過程中線端點
  接往變電所時的延伸與吸附，本 notebook 未逐條拆解。
- **raw 有 677 筆線（280 km）完全沒有電壓標註**，在 raw 側被歸入「其他」，
  clean 側已推定電壓或丟棄，這是「其他」類比值偏低的主因。

結論：**三個主要電壓等級在清理前後的長度一致（偏差 2–8%），沒有整段網路憑空消失或新增。**

In [ ]:
compare = pd.DataFrame({
    "OSM raw 迴路長度_km": raw_by_v["迴路長度_km"],
    "OSM clean 迴路長度_km": clean_by_v["迴路長度_km"],
})
compare["clean/raw 比值"] = (
    compare["OSM clean 迴路長度_km"] / compare["OSM raw 迴路長度_km"]
).replace([np.inf, -np.inf], np.nan)

K.FIG_DIR.mkdir(parents=True, exist_ok=True)
compare.round(3).to_csv(K.FIG_DIR / "01_table_line_length_by_voltage.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "01_table_line_length_by_voltage.csv")
compare.round(2)

In [ ]:
plot_df = compare.drop(index="合計")[["OSM raw 迴路長度_km", "OSM clean 迴路長度_km"]]

fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot.bar(
    ax=ax, width=0.75,
    color=["#b0b0b0", "#2b8cbe"],
    edgecolor="white", linewidth=0.8,
)
ax.set_title("韓國輸電線路迴路長度（按電壓等級，EPSG:5179）", fontsize=13, pad=10)
ax.set_ylabel("迴路長度 [km]")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(["OSM raw", "OSM clean"], frameon=False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{int(x):,}"))

for cont in ax.containers:
    ax.bar_label(cont, fmt="%.0f", fontsize=8, padding=2)

K.savefig("01_fig2_line_length_by_voltage")
plt.show()

### 1.2.5 模型實際採用的線型參數

`config.yaml` 的 `lines.ac_types.default` 補上了韓國的 154 / 345 / 765 kV。
上游 `_helpers.py` 的 `get_linetype_by_voltage_and_country()` 會自動挑「最接近的」
線型，如果不補這三個等級，154 會被當成 132、345 當成 380、765 當成 750，阻抗會失真。
下表是模型實際使用的對應關係。

In [ ]:
ac_types = K.CONFIG["lines"]["ac_types"]["default"]
kr_voltages = [154.0, 345.0, 765.0]

line_types = pd.DataFrame({
    "電壓等級 [kV]": [f"{int(v)}" for v in kr_voltages],
    "config 指定線型": [ac_types[v] for v in kr_voltages],
    "若未補值會誤用": ["132 kV 線型", "380 kV 線型", "750 kV 線型"],
})
print("config.yaml 的 electricity.voltages：", K.CONFIG["electricity"]["voltages"])
line_types

### 1.2.6 對照值說明

**無公開對照值。** 非洲原版在這裡拿 NamPower 年報（400 kV 1,179 km、330 kV 522 km …）
與 World Bank 非洲電網圖層做對照。韓國的對應資料（KEPCO 輸電線路長度統計）
**不在本 repo 內**，本 notebook 因此不列任何官方數字，也不從記憶或網路填補。

若之後要補，需要把 KEPCO 官方統計（例如《한국전력통계》輸電設備章節的
架空線路 亘長/條長 按 765/345/154 kV 分列）放進 repo，再於此處讀檔比對。
在那之前，1.2.4 的 raw 對 clean 自比是本節唯一有效的驗證。